In [1]:
from typing import Dict

In [2]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

In [3]:
from sqlitesearch import TextSearchIndex

index = TextSearchIndex(
    text_fields=['question', 'section', 'answer'],
    keyword_fields=['course'],
    db_path='faq.db'
)

def search(query: str) -> Dict[str, str]:
    """
    Search the FAQ database for relevant information
    """
    return index.search(
        query,
        boost_dict={'question': 3.0, 'section': 0.5},
        filter_dict={'course': 'llm-zoomcamp'},
        num_results=5
    )

In [4]:
search_tool = {
    'type': 'function',
    'name': 'search',
    'description': 'Search the FAQ database for relevant information',
    'parameters': {
        'type': 'object',
        'properties': {
            'query': {
                'type': 'string',
                'description': 'Search query text to look up in the course FAQ.'
            }
        },
        'required': ['query'],
        'additionalProperties': False
    }
}

In [5]:
import json

def make_call(call):
    args = json.loads(call.arguments)

    results = search(**args)
    results_json = json.dumps(results, indent=2)

    return {
        'type': 'function_call_output',
        'call_id': call.call_id,
        'output': results_json,
    }

In [6]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results, and then refine your search with new keywords.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

question = 'I just discovered the course. Can I join now?'

In [ ]:
def loop(instructions:str, question:str, model:str='gpt-5.4-mini') -> str:
    
    messages = [
                    {'role': 'developer', 'content': instructions},
                    {'role': 'user', 'content': question},
                ]

    it = 1

    while True:

        print(f'Iteration number {it}')
        has_function_calls = False

        response = openai_client.responses.create(
            model = model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:

            if item.type == 'function_call':
                has_function_calls = True
                print('function_call:', item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == 'message':
                last_answer = item.content[0].text
                print('ASSISTANT:')
                print(item.content[0].text)

        it += 1
        if not has_function_calls:
            break

    return last_answer

In [8]:
result = loop(instructions, question)

Iteration number 1
function_call: search {"query":"join course now late enrollment discovered the course can I join now"}
function_call: search {"query":"course enrollment late join after start discovered the course can I join now"}
Iteration number 2
ASSISTANT:
Yes — you can still join the course.

If your goal is to get a certificate, the key thing is to submit your project while submissions are still open. Also, certificates are only available for the live cohort, not self-paced participation.

If you want, I can also help with:
- whether you need to register,
- how homework/submissions work,
- or what you need for the certificate.


In [1]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [10]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [12]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

In [14]:
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model='gpt-5.4-mini')
)

In [15]:
result = runner.loop(
    prompt='How do I run Ollama locally?',
    callback=callback
)

-> Response received


-> Response received


In [17]:
result.cost

CostInfo(input_cost=Decimal('0.00269175'), output_cost=Decimal('0.0015975'), total_cost=Decimal('0.00428925'))